In [91]:
# Import necessary libraries
%load_ext autoreload
%autoreload 2

import sys
import os
import math

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

# Import Mars library components
import mars
from mars import spin_model, spectra_manager, mesher, constants
from mars import utils
from mars import population

# # Let's import some useful functions for plotting simulation results over time.
from mars import visualization

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [92]:
dtype = torch.float64
device = torch.device("cpu")

In [93]:
# Let's start from the sample creation
g_tensor = spin_model.Interaction([2.00, 2.01, 2.02], dtype=dtype, device=device)
zfs_interaction = spin_model.DEInteraction([140e6, 40e6], dtype=dtype, device=device)  # 500 and 100 MHz

base_spin_system = spin_model.SpinSystem(
    electrons=[1.0],  # S=1 triplet
    g_tensors=[g_tensor],
    electron_electron=[(0, 0, zfs_interaction)]
)

triplet = spin_model.MultiOrientedSample(
    base_spin_system=base_spin_system,
    ham_strain=0.0,
    gauss=0.0002,
    lorentz=0.0002,
    device=device,
    mesh=(20, 20),
    dtype=dtype
)

In [104]:
field_free, field_dep = triplet.get_librations_along_axis(torch.tensor([1.0, 0.0, 0.0], dtype=dtype))

torch.Size([3, 3])
tensor([[ 0.0000+0.j,  0.0000+0.j,  0.0000+0.j],
        [ 0.0000+0.j,  0.0000+0.j, -0.0100+0.j],
        [ 0.0000+0.j, -0.0100+0.j,  0.0000+0.j]], dtype=torch.complex128)


In [109]:
field_free[34] ** 2 * (2 * math.pi)

tensor([[ 0.0000e+00+0.j, -1.0179e+17+0.j,  0.0000e+00+0.j],
        [-1.0179e+17-0.j,  0.0000e+00+0.j, -1.0179e+17-0.j],
        [ 0.0000e+00+0.j, -1.0179e+17+0.j,  0.0000e+00+0.j]],
       dtype=torch.complex128)

In [65]:
field_dep[50]

tensor([[0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j, 0.+0.j]], dtype=torch.complex128)

In [6]:
class SpectralDensity():
    def __init__(self, w_k, temperature):
        w_k_rad = constants.unit_converter(w_k, "cm-1_to_Hz") / (2 * math.pi)
        exp = torch.exp(torch.tensor(constants.unit_converter(w_k, "cm-1_to_K") / temperature))
        self._occup = 1 / (exp - 1)
        self._delta = torch.sqrt(w_k_rad ** 2 * exp * self._occup ** 2)
        self.w_k_rad = w_k_rad
        
        
    def __call__(self, w_rad):
        plus = self.w_k_rad + w_rad
        minus = self.w_k_rad - w_rad
        
        return self._lorentz(plus) * self._occup + self._lorentz(minus) * (self._occup + 1)
        
    def _lorentz(self, arg):
        return self._delta / (self._delta ** 2 + arg ** 2)
        

In [118]:
density = SpectralDensity(150, 200)

In [119]:
density(10e9)

tensor(1.4183e-12)